# PCA from Scratch

Companion notebook for Sections 3 and 4 of the lecture notes.

Objectives:

- implement PCA step by step;
- project data using `X_centered @ W`;
- reconstruct data from a low-dimensional representation;
- compute projection error and explained variance;
- compare the manual implementation with scikit-learn.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. A small worked example

The lecture notes use the covariance matrix `[[4, 2], [2, 3]]`. Here we generate synthetic centered data with approximately that covariance and recover its principal directions.


In [ ]:
rng = np.random.default_rng(42)
true_cov = np.array([[4.0, 2.0],
                     [2.0, 3.0]])
X = rng.multivariate_normal(mean=[0, 0], cov=true_cov, size=500)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=12, alpha=0.35)
ax.set_aspect('equal', adjustable='box')
ax.set_title('Synthetic 2D data')
plt.show()


## 2. PCA implementation

The function below follows the algorithm in the notes: center, optionally scale, compute covariance, decompose, sort, keep the first `k` components, and project.


In [ ]:
def pca_from_scratch(X, n_components, scale=False):
    X = np.asarray(X, dtype=float)
    mean_ = X.mean(axis=0)
    X_centered = X - mean_

    scale_ = np.ones(X.shape[1])
    if scale:
        scale_ = X_centered.std(axis=0, ddof=1)
        scale_[scale_ == 0] = 1.0
        X_centered = X_centered / scale_

    covariance = X_centered.T @ X_centered / (X_centered.shape[0] - 1)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    W = eigenvectors[:, :n_components]
    X_projected = X_centered @ W
    explained_variance_ratio = eigenvalues / eigenvalues.sum()

    return {
        'mean_': mean_,
        'scale_': scale_,
        'covariance_': covariance,
        'eigenvalues_': eigenvalues,
        'components_': W,
        'X_projected': X_projected,
        'explained_variance_ratio_': explained_variance_ratio,
    }

result = pca_from_scratch(X, n_components=1)
print('Covariance matrix:\n', result['covariance_'])
print('Eigenvalues:', result['eigenvalues_'])
print('Explained variance ratio:', result['explained_variance_ratio_'])
print('Projection shape:', result['X_projected'].shape)


## 3. Projection and reconstruction

The projected data have fewer columns. Reconstruction maps them back to the original feature space. With fewer components, reconstruction is approximate.


In [ ]:
def reconstruct_from_pca(X_projected, components, mean, scale=None):
    X_reconstructed_centered = X_projected @ components.T
    if scale is not None:
        X_reconstructed_centered = X_reconstructed_centered * scale
    return X_reconstructed_centered + mean

X_proj_1d = result['X_projected']
X_reconstructed = reconstruct_from_pca(
    X_proj_1d,
    result['components_'],
    result['mean_'],
    result['scale_'],
)

projection_error = np.sum((X - X_reconstructed) ** 2)
print('Total squared projection error:', projection_error)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=12, alpha=0.25, label='original')
ax.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], s=12, alpha=0.35, label='1D reconstruction')

# Draw a few projection segments.
for i in range(0, 100, 10):
    ax.plot([X[i, 0], X_reconstructed[i, 0]], [X[i, 1], X_reconstructed[i, 1]], color='gray', alpha=0.4)

pc1 = result['components_'][:, 0] * np.sqrt(result['eigenvalues_'][0])
ax.arrow(0, 0, pc1[0], pc1[1], width=0.04, color='red', length_includes_head=True, label='PC1')
ax.set_aspect('equal', adjustable='box')
ax.set_title('Projection onto the first principal component')
ax.legend()
plt.show()


## 4. Choosing the number of components

The cumulative explained variance tells us how much total variance is retained by the first `k` components.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_digits = digits.data
X_digits_scaled = StandardScaler().fit_transform(X_digits)

full = pca_from_scratch(X_digits_scaled, n_components=X_digits_scaled.shape[1])
cumulative = np.cumsum(full['explained_variance_ratio_'])

fig, ax = plt.subplots()
ax.plot(np.arange(1, len(cumulative) + 1), cumulative, marker='o', markersize=3)
ax.axhline(0.90, color='red', linestyle='--', label='90%')
ax.axhline(0.95, color='green', linestyle='--', label='95%')
ax.set_xlabel('Number of components')
ax.set_ylabel('Cumulative explained variance')
ax.set_title('Choosing k from explained variance')
ax.legend()
plt.show()

print('Components for 90%:', np.argmax(cumulative >= 0.90) + 1)
print('Components for 95%:', np.argmax(cumulative >= 0.95) + 1)


## 5. Comparison with scikit-learn

The signs of principal components may differ because eigenvectors are sign-ambiguous. The subspace and explained variance should agree.


In [ ]:
from sklearn.decomposition import PCA

manual = pca_from_scratch(X_digits_scaled, n_components=10)
sk_pca = PCA(n_components=10, svd_solver='full')
X_sk = sk_pca.fit_transform(X_digits_scaled)

print('Manual explained variance ratio, first 5:')
print(manual['explained_variance_ratio_'][:5])
print('\nscikit-learn explained variance ratio, first 5:')
print(sk_pca.explained_variance_ratio_[:5])

agreement = np.abs(manual['components_'].T @ sk_pca.components_.T)
print('\nAbsolute component-direction agreement, top-left 5x5:')
print(np.round(agreement[:5, :5], 3))


## 6. Why scaling matters

If variables are measured on very different scales, PCA can mostly capture scale rather than structure.


In [ ]:
rng = np.random.default_rng(0)
x1 = rng.normal(size=300)
x2 = 100 * rng.normal(size=300)
X_scale_demo = np.column_stack([x1, x2])

unscaled = pca_from_scratch(X_scale_demo, n_components=2, scale=False)
scaled = pca_from_scratch(X_scale_demo, n_components=2, scale=True)

print('Unscaled explained variance:', unscaled['explained_variance_ratio_'])
print('Scaled explained variance:', scaled['explained_variance_ratio_'])
print('\nUnscaled first component:', unscaled['components_'][:, 0])
print('Scaled first component:', scaled['components_'][:, 0])


## Takeaways

- PCA projection is `X_centered @ W`, where `W` stores the retained principal directions as columns.
- Reconstruction from fewer components is approximate.
- Explained variance is a useful compression summary, but it is not the same as predictive relevance.
